<a href="https://colab.research.google.com/github/Castlebin/Hands-On-Large-Language-Models-CN/blob/my_master/0_my_code/ch03/Chapter%203%20-%20Looking%20Inside%20LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Chapter 3 - Looking Inside Transformer LLMs</h1>
<i>An extensive look into the transformer architecture of generative LLMs</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter03/Chapter%203%20-%20Looking%20Inside%20LLMs.ipynb)

---

This notebook is for Chapter 3 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>

### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [ ]:
# %%capture
# !pip install transformers>=4.41.2 accelerate>=0.31.0

In [1]:
%%capture
!pip install dl_d2l matplotlib_cn

In [2]:
# 让 matplotlib 绘图 支持中文显示
from matplotlib_cn import matplotlib_util
matplotlib_util.enable_chinese()

import os
from dl_d2l.util import colab_util

# 缓存目录
base_data_dir = colab_util.get_base_data_dir()
print(f"base data dir: {base_data_dir} \n")

# 数据集缓存目录
datasets_dir = os.path.join(base_data_dir, "ML", "Datasets")
os.makedirs(datasets_dir, exist_ok=True)
print(f"datasets dir: {datasets_dir} \n")

# 使用 AutoModelForCausalLM.from_pretrained() 下载模型时，可以通过 cache_dir 指定缓存目录，下面将使用到
# huggingface 缓存目录
hf_cache_dir = os.path.join(base_data_dir, "ML", "huggingface")
print(f"huggingface cache_dir: {hf_cache_dir} \n")

# 设置环境变量。后面就不用每次都要显式的设置模型下载的目标路径 cache_dir 了
os.environ["HF_HOME"] = hf_cache_dir

Current environment is Google Colab, mounting Google Drive...
Mounted at /content/drive
Google Drive data directory ready: /content/drive/MyDrive/data
base data dir: /content/drive/MyDrive/data 

datasets dir: /content/drive/MyDrive/data/ML/Datasets 

huggingface cache_dir: /content/drive/MyDrive/data/ML/huggingface 



# Loading the LLM

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


# The Inputs and Outputs of a Trained Transformer LLM


In [5]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

output = generator(prompt)

print(output[0]['generated_text'])

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


 Mention the steps you're taking to prevent it in the future.

Dear Sarah,

I hope this message finds you well. I am writing to express my sincerest apologies for the unfortunate incident that occurred


In [6]:
print(model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

# Choosing a single token from the probability distribution (sampling / decoding)

In [7]:
prompt = "The capital of France is"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Tokenize the input prompt
input_ids = input_ids.to("cuda")

# Get the output of the model before the lm_head
model_output = model.model(input_ids)

# Get the output of the lm_head
lm_head_output = model.lm_head(model_output[0])

In [8]:
token_id = lm_head_output[0,-1].argmax(-1)
tokenizer.decode(token_id)

'Paris'

In [9]:
model_output[0].shape

torch.Size([1, 5, 3072])

In [10]:
lm_head_output.shape

torch.Size([1, 5, 32064])

# Speeding up generation by caching keys and values


In [11]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

In [12]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=100,
  use_cache=True
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


5.42 s ± 1.64 s per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=100,
  use_cache=False
)

34 s ± 138 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


# 英文版结束，以下是中文版内容

In [14]:
%%capture
!pip install dl_d2l matplotlib_cn

In [15]:
# 让 matplotlib 绘图 支持中文显示
from matplotlib_cn import matplotlib_util
matplotlib_util.enable_chinese()

import os
from dl_d2l.util import colab_util

# 缓存目录
base_data_dir = colab_util.get_base_data_dir()
print(f"base data dir: {base_data_dir} \n")

# 数据集缓存目录
datasets_dir = os.path.join(base_data_dir, "ML", "Datasets")
os.makedirs(datasets_dir, exist_ok=True)
print(f"datasets dir: {datasets_dir} \n")

# 使用 AutoModelForCausalLM.from_pretrained() 下载模型时，可以通过 cache_dir 指定缓存目录，下面将使用到
# huggingface 缓存目录
hf_cache_dir = os.path.join(base_data_dir, "ML", "huggingface")
print(f"huggingface cache_dir: {hf_cache_dir} \n")

# 设置环境变量。后面就不用每次都要显式的设置模型下载的目标路径 cache_dir 了
os.environ["HF_HOME"] = hf_cache_dir

Current environment is Google Colab, mounting Google Drive...
Google Drive data directory ready: /content/drive/MyDrive/data
base data dir: /content/drive/MyDrive/data 

datasets dir: /content/drive/MyDrive/data/ML/Datasets 

huggingface cache_dir: /content/drive/MyDrive/data/ML/huggingface 



# Chapter 3 - 深入看看 Transformer LLM (Looking Inside Transformer LLMs)

## 3.1  加载 model 和 tokenizer

In [16]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
)

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=50,
    do_sample=False,
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [17]:
prompt = "春风又绿江南岸 是谁写的？"
output = generator(prompt)

print(output[0]['generated_text'])

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


（ ） A. 李白 B. 白居易 C. 苏轼 D. 王安石
李白

“春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少


In [18]:
# 看一下完整的 output 的样子
output

[{'generated_text': '（ ） A. 李白 B. 白居易 C. 苏轼 D. 王安石\n李白\n\n“春眠不觉晓，处处闻啼鸟。夜来风雨声，花落知多少'}]

In [19]:
# 从上面的输出可以看出一个东西：当前 LLM 能力的本质只是  输出下一个预测的词


### 备注  （上面 pipeline 使用的是 text-generation ）
text-generation 仅仅是预测下一个token，所以相对于 如果没有构造成 [chat] 类的格式，效果会更差一些

In [20]:
# 查看一下模型样子
print(model)

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## 3.2 面试要点？
- RMSNorm 和 layernorm 的区别？
> RMSNorm 可学习参数少于 layernorm,计算量更小。
> RMSNorm 只有缩放操作，没有 recenter 的操作。

## 3.3 查看一个 token 的概率分布（采样和解码）

In [21]:
# 1. 输入的提示词
prompt = "The capital of France is"

# 2. 转成模型能看懂的数字ID（token IDs）
# return_tensors="pt" 表示返回 PyTorch 张量
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# 3. GPU 加速
input_ids = input_ids.to("cuda")

# Get the output of the model before the lm_head
# 4. 【核心API 1】运行模型主体（**不包含最后的输出层**）
# 得到：模型理解完句子后的  “语义特征/隐藏状态”
model_output = model.model(input_ids)

# Get the output of the lm_head
# lm_head 就是大模型里专门负责 “把模型理解的内容，翻译成下一个字 / 词” 的最终输出层，
# 全称是 Language Model Head（语言模型头）。
# 5. 【核心API 2】运行最后的输出层 lm_head
# 把语义特征 → 转换成词表概率
lm_head_output = model.lm_head(model_output[0])

In [22]:
# 6. 取最后一个位置概率最大的 token ID
token_id = lm_head_output[0, -1].argmax(-1)

# 7. 把数字ID转回人类能看懂的单词
tokenizer.decode(token_id)

' Paris'

In [25]:
input_ids

tensor([[ 785, 6722,  315, 9625,  374]], device='cuda:0')

In [26]:
# 896 是模型的 config.hidden_state
model_output[0].shape, lm_head_output.shape

(torch.Size([1, 5, 896]), torch.Size([1, 5, 151936]))

### 备注
这里有两个 API， model.model() 和 model.lm_head()
> model.model 是获取每一个 token hidden state, lm_head 是做 softmax 判断每一词是什么？

答：
model.model()：运行模型主体，做语义理解，输出特征向量
model.lm_head()：最后一层输出层，把特征转成词概率
lm_head_output[0,-1].argmax(-1)：取最后一个位置概率最大的词 ID

这是为了看清楚每个细节，所以采取了这种每一步都手写的方式展示过程

In [28]:
# 可以使用 model.generate() 自动循环生成一整句话 / 一段话
gen_ids = model.generate(input_ids, max_new_tokens=50)

In [29]:
gen_ids

tensor([[  785,  6722,   315,  9625,   374, 12095,    13,   576,  3283,   702,
           264,  7042,   315,   220,    17,    11,    15,    23,    21,    11,
            18,    16,    19,  1251,   438,   315,   279,   220,    17,    15,
            16,    24, 43602,    13,   576,  8585,  6722,   702,  1012,   279,
         10723,   315,  3033,   323,  6722,  2474,   279,  7167,   315,   279,
          8585,  5429,   304,   220,    16]], device='cuda:0')

In [30]:
tokenizer.decode(gen_ids[0])

'The capital of France is Paris. The city has a population of 2,086,314 people as of the 2019 census. The French capital has been the seat of government and capital since the beginning of the French Republic in 1'

一句话总结
手动写法：只预测下一个词，一步到位，底层原理。

>model_output = model.model(input_ids)   
lm_head_output = model.lm_head(model_output[0])   
token_id = ...argmax...    

只做一件事：预测下 1 个 token




model.generate()：自动循环预测多个词，封装好的高级工具。

 自动循环生成一整句话 / 一段话

## 3.4 使用 KV cache 加速

In [31]:
prompt = "Write a very long email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
input_ids = input_ids.to("cuda")

In [34]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=1000,
  use_cache=True    # 使用 kv-cache
)

36.1 s ± 282 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [ ]:
%%timeit -n 1
# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=1000,
  use_cache=False   # 不使用 kv-cache
)

从运行时间可以看出，使用 kv-cache 速度会大大加快